# 03: Libraries and Packages

One of Python's superpowers is its ecosystem — tens of thousands of packages built by other people that you can use for free. Data analysis, plotting, statistics, machine learning, API calls — someone already wrote the hard parts.

This notebook covers how to find, install, and use packages.

> **References:**  
> - [Think Python — Chapter 15: Classes and Objects](https://allendowney.github.io/ThinkPython/chap15.html) (Objects and modules)  
> - [MIT Missing Semester — Lecture 4: Data Wrangling](https://missing.csail.mit.edu/2020/data-wrangling/) (Processing and transforming data with command-line and scripting tools)

> **Navigation:** [← Previous: Working with Files](../01-python-foundations/02-working-with-files.ipynb) | [Module Index](../README.md) | [Next: Data Structures in Practice →](../01-python-foundations/04-data-structures-in-practice.ipynb)

> **Why this matters for your work**
> 
> - You don't write everything from scratch — and neither does Claude. When Claude generates analysis code, it uses libraries like pandas, numpy, and matplotlib. If you don't understand `import pandas as pd`, you can't read or trust what Claude produces.
> - The `anthropic` package is how you'll call the Claude API from Python. Understanding how packages work means you can install and use any tool Claude suggests — from BioPython for sequence analysis to scipy for curve fitting your dose-response data.
> - Every computational protein design tool in your pipeline (RFdiffusion, ProteinMPNN, AlphaFold2) is a Python package. Understanding imports and virtual environments is what lets you actually run them.
> - numpy array operations are how calcium imaging traces, binding curves, and electrophysiology data get analyzed efficiently — one line instead of a hundred-iteration loop.

---

## What are packages and why they matter

A **package** (or **library**) is a bundle of pre-written Python code. Instead of writing everything from scratch, you import what you need.

Analogy: You don't synthesize your own antibodies for every western blot. You buy validated ones from a supplier. Python packages work the same way — battle-tested tools you plug into your code.

Here are the packages you'll use most in this tutorial:

| Package | What it does | Your use case |
|---------|-------------|---------------|
| **pandas** | Tables and data manipulation | Analyzing screening results, RNA-seq data |
| **numpy** | Fast math on arrays of numbers | Calcium imaging traces, numerical analysis |
| **matplotlib** | Plotting and visualization | Dose-response curves, expression plots |
| **seaborn** | Statistical plots (built on matplotlib) | Heatmaps, violin plots, pair plots |
| **anthropic** | Claude API client | Sending prompts to Claude from Python |

---

## Importing packages

There are a few ways to import. Let's see them all.

In [ ]:
# Style 1: Import the whole package
import math

print(math.sqrt(144))   # square root
print(math.log10(1000)) # log base 10

In [ ]:
# Style 2: Import specific things from a package
from math import sqrt, pi

print(sqrt(144))  # no need to write math.sqrt
print(pi)

In [ ]:
# Style 3: Import with an alias (nickname)
# These aliases are universal conventions — everyone uses them
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print(f"pandas version:  {pd.__version__}")
print(f"numpy version:   {np.__version__}")

**Which style should you use?** For the big data science packages, always use the standard aliases:

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
```

These are so universal that if you write `pd.read_csv()`, any Python user (or AI assistant) will immediately know what you mean.

```mermaid
graph TB
    subgraph "How Python Imports Work"
        YOUR["YOUR CODE<br/><code>import pandas as pd</code>"]

        subgraph "Third-Party Packages (pip install)"
            PD["pandas"]
            NP["numpy"]
            PLT["matplotlib"]
            SNS["seaborn"]
            ANT["anthropic"]
        end

        subgraph "Standard Library (comes with Python)"
            CSV["csv"]
            JSON["json"]
            MATH["math"]
            PATH["pathlib"]
            OS["os"]
        end

        subgraph "Python Built-ins (always available)"
            PRINT["print()"]
            LEN["len()"]
            RANGE["range()"]
            TYPE["type()"]
        end

        YOUR --> PD
        YOUR --> NP
        YOUR --> CSV
        YOUR --> JSON
        YOUR --> PRINT
    end

    style YOUR fill:#cc4444,color:#fff
    style PD fill:#4488cc,color:#fff
    style NP fill:#4488cc,color:#fff
    style PLT fill:#4488cc,color:#fff
    style SNS fill:#4488cc,color:#fff
    style ANT fill:#4488cc,color:#fff
    style CSV fill:#44aa88,color:#fff
    style JSON fill:#44aa88,color:#fff
    style MATH fill:#44aa88,color:#fff
    style PATH fill:#44aa88,color:#fff
    style OS fill:#44aa88,color:#fff
    style PRINT fill:#cc8844,color:#fff
    style LEN fill:#cc8844,color:#fff
    style RANGE fill:#cc8844,color:#fff
    style TYPE fill:#cc8844,color:#fff
```

Three layers of Python code you can use:
- **Built-ins** (orange) -- always available, no import needed
- **Standard library** (green) -- comes with Python, just `import` it
- **Third-party packages** (blue) -- installed with `pip`, then `import`

---

## Quick tour: numpy — fast math on arrays

NumPy is the foundation for all numerical computing in Python. Its core object is the **array** — like a Python list, but optimized for math.

In [ ]:
import numpy as np

# Calcium fluorescence ratios from a DRG neuron (mock data)
# Each value is F/F0 at one time point during a calcium imaging experiment
trace = np.array([1.0, 1.02, 0.98, 1.01, 1.05, 1.8, 3.2, 4.1, 3.8, 2.9, 
                   2.1, 1.6, 1.3, 1.1, 1.05, 1.02, 0.99, 1.01, 1.0, 0.98])

print(f"Number of time points: {len(trace)}")
print(f"Baseline (mean of first 4): {trace[:4].mean():.3f}")
print(f"Peak F/F0: {trace.max():.2f}")
print(f"Peak time point: {trace.argmax()}")
print(f"Mean: {trace.mean():.2f}")
print(f"Std dev: {trace.std():.2f}")

NumPy lets you do math on entire arrays at once — no loops needed.

- `trace.max()` — find the peak
- `trace.argmax()` — find *where* the peak is
- `trace[:4]` — grab the first 4 elements (slicing)
- `trace[:4].mean()` — mean of those 4 elements

In [ ]:
import numpy as np

# Math on whole arrays — no loops
trace = np.array([1.0, 1.02, 0.98, 1.01, 1.05, 1.8, 3.2, 4.1, 3.8, 2.9, 
                   2.1, 1.6, 1.3, 1.1, 1.05, 1.02, 0.99, 1.01, 1.0, 0.98])

baseline = trace[:4].mean()

# Calculate delta F / F0 for every point at once
delta_f = (trace - baseline) / baseline

print("Delta F/F0 values:")
print(np.round(delta_f, 3))

That `(trace - baseline) / baseline` applied the formula to all 20 values simultaneously. In plain Python with lists, you'd need a loop. NumPy makes this kind of array math effortless.

---

## Quick tour: matplotlib — plotting

Matplotlib is Python's main plotting library. The syntax takes some getting used to, but once you know the basics, you can make any plot.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Mock calcium imaging data: time in seconds, F/F0 trace
time_s = np.arange(0, 20, 1)  # 0, 1, 2, ... 19 seconds
trace = np.array([1.0, 1.02, 0.98, 1.01, 1.05, 1.8, 3.2, 4.1, 3.8, 2.9, 
                   2.1, 1.6, 1.3, 1.1, 1.05, 1.02, 0.99, 1.01, 1.0, 0.98])

plt.figure(figsize=(8, 4))
plt.plot(time_s, trace, color="#2563eb", linewidth=2)
plt.axhline(y=1.0, color="gray", linestyle="--", alpha=0.5, label="Baseline")
plt.axvspan(4.5, 5.5, color="red", alpha=0.15, label="Capsaicin pulse")
plt.xlabel("Time (s)")
plt.ylabel("F / F₀")
plt.title("DRG Neuron Calcium Response — TRPV1 Activation")
plt.legend()
plt.tight_layout()
plt.show()

The pattern is always:
1. `plt.figure()` — create the canvas
2. `plt.plot()` (or `.bar()`, `.scatter()`, etc.) — draw the data
3. `plt.xlabel()`, `plt.ylabel()`, `plt.title()` — label things
4. `plt.show()` — display the result

You don't need to memorize every option. When you need to customize a plot, just describe what you want to Claude and it'll give you the code.

---

## Quick tour: pandas — data in tables

We previewed pandas in the last notebook. Here's a bit more of what it can do.

In [ ]:
import pandas as pd
from pathlib import Path

# Read the binder screening CSV
csv_file = Path.cwd() / ".." / "01-python-foundations" / "sample_data" / "binder_screening.csv"
# If running from the same directory as notebook 02:
csv_file = Path.cwd() / "sample_data" / "binder_screening.csv"

df = pd.read_csv(csv_file)

print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns\n")
print("Columns:", list(df.columns))
print()
df.head()

In [ ]:
# Group by target and get average Kd
summary = df.groupby("target")["predicted_kd_nm"].agg(["mean", "min", "count"])
summary.columns = ["Mean Kd (nM)", "Best Kd (nM)", "# Candidates"]
summary

In [ ]:
# Sort by affinity (best first)
df.sort_values("predicted_kd_nm").head(5)

We'll go deep on pandas in Module 06 (Data Skills). For now, just notice how much you can do with very little code.

---

## pip: installing packages

Packages come from the **Python Package Index** ([PyPI](https://pypi.org), pronounced "pie-pee-eye"). You install them with `pip`.

We already installed the core packages when setting up this tutorial, but here's how it works:

```bash
# Install a single package
pip install pandas

# Install a specific version
pip install pandas==2.2.0

# Install multiple packages
pip install pandas numpy matplotlib

# See what's installed
pip list

# See what's installed and save it to a file (for reproducibility)
pip freeze > requirements.txt
```

> **Why virtual environments matter** ([MIT Missing Semester](https://missing.csail.mit.edu/2020/course-shell/) concept):  
> Different projects may need different package versions. A **virtual environment** ([`venv`](https://docs.python.org/3/library/venv.html)) is an isolated Python setup for each project. Our `ai-tutorial` kernel uses the `.venv/` directory — packages you install here won't affect any other Python projects on your machine. See [Python docs: `pip`](https://pip.pypa.io/en/stable/) for more on package management.

In [ ]:
# You can run pip from inside a notebook using the ! prefix (runs a shell command)
# Let's see what packages are installed in our environment
!pip list --format=columns 2>/dev/null | head -20

---

## How to find and learn packages

When you need to do something in Python, the workflow is:

1. **Ask Claude:** "What's the best Python package for [task]?" — this is the fastest way
2. **Search PyPI:** https://pypi.org — the official package repository
3. **Check the docs:** Every good package has documentation with examples

You can also explore a package from within Python:

In [ ]:
import numpy as np

# See what functions are available
# (This prints a lot — just scan it to get a feel)
print([x for x in dir(np) if not x.startswith("_")][:30])  # first 30 public names

In [ ]:
# Get help on a specific function
help(np.mean)

Honestly? You'll rarely read docs line-by-line. The real workflow is:
1. Know the package exists
2. Know roughly what it does
3. Ask Claude (or search) for the specific function you need

That's a legitimate professional workflow, not cheating.

---

## Quick tour: seaborn — beautiful statistical plots

Seaborn builds on matplotlib and makes statistical visualizations easy. Let's build up a realistic example.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Generate mock calcium imaging data: % of DRG neurons responding
# across different binder concentrations
np.random.seed(42)  # makes random numbers reproducible

concentrations = [0, 1, 10, 100, 1000]  # nM
n_replicates = 4

rows = []
for conc in concentrations:
    for rep in range(n_replicates):
        # Simulate a dose-response: more binder → more neurons inhibited
        if conc == 0:
            pct_responding = np.random.normal(78, 5)  # baseline ~78% respond to capsaicin
        else:
            # Dose-dependent inhibition with some noise
            inhibition = 65 * (conc / (conc + 50))  # Hill-ish curve, EC50 ~50 nM
            pct_responding = np.random.normal(78 - inhibition, 4)
        
        rows.append({
            "concentration_nM": conc,
            "replicate": rep + 1,
            "pct_responding": max(0, min(100, pct_responding))  # clamp to 0-100
        })

dose_response = pd.DataFrame(rows)
dose_response.head(10)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Recreate the data (notebook is self-contained)
np.random.seed(42)
concentrations = [0, 1, 10, 100, 1000]
rows = []
for conc in concentrations:
    for rep in range(4):
        if conc == 0:
            pct = np.random.normal(78, 5)
        else:
            inhibition = 65 * (conc / (conc + 50))
            pct = np.random.normal(78 - inhibition, 4)
        rows.append({"concentration_nM": conc, "pct_responding": max(0, min(100, pct))})
dose_response = pd.DataFrame(rows)

# Plot: dose-response with individual data points
fig, ax = plt.subplots(figsize=(8, 5))

# Use log scale for concentration (add small offset for 0 nM)
dose_response["conc_label"] = dose_response["concentration_nM"].apply(
    lambda x: str(int(x)) if x > 0 else "Vehicle"
)

order = ["Vehicle", "1", "10", "100", "1000"]

sns.barplot(data=dose_response, x="conc_label", y="pct_responding",
            order=order, color="#93c5fd", edgecolor="#1e40af", alpha=0.7, ax=ax)
sns.stripplot(data=dose_response, x="conc_label", y="pct_responding",
              order=order, color="#1e40af", size=6, alpha=0.8, ax=ax)

ax.set_xlabel("DNB-Nav17-003 Concentration (nM)")
ax.set_ylabel("% DRG Neurons Responding to Capsaicin")
ax.set_title("Binder Dose-Response: Inhibition of Capsaicin-Evoked Calcium Transients")
ax.set_ylim(0, 100)

plt.tight_layout()
plt.show()

That's a publication-quality dose-response figure in about 15 lines of code. Let's break down what happened:

- `sns.barplot()` — draws the bars (means with confidence intervals)
- `sns.stripplot()` — overlays individual data points
- `ax.set_xlabel()`, etc. — labels and formatting

The key insight: **seaborn works directly with pandas DataFrames**. You say "x is this column, y is that column" and it figures out the rest.

---

## Combining packages: a mini-analysis

Let's put it all together. Here's a realistic scenario: you have calcium imaging traces from multiple neurons and want to identify responders.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Simulate calcium traces for 6 DRG neurons
# 30 time points, 1 second each. Stimulus at t=10s.
np.random.seed(123)
time_s = np.arange(0, 30, 1)
n_neurons = 6

traces = {}
for i in range(n_neurons):
    # Baseline with noise
    baseline = 1.0 + np.random.normal(0, 0.03, 30)
    
    # Some neurons respond (peak at t=12), some don't
    is_responder = i in [0, 2, 4, 5]  # 4 of 6 respond
    if is_responder:
        peak_height = np.random.uniform(2.0, 5.0)
        # Simple response shape: rapid rise, exponential decay
        response = np.zeros(30)
        for t in range(10, 30):
            response[t] = peak_height * np.exp(-(t - 11) / 5)
        baseline += response
    
    traces[f"Neuron {i+1}"] = baseline

# Plot all traces
fig, axes = plt.subplots(2, 3, figsize=(12, 6), sharex=True, sharey=True)

for idx, (name, trace) in enumerate(traces.items()):
    ax = axes[idx // 3][idx % 3]
    ax.plot(time_s, trace, color="#2563eb", linewidth=1.5)
    ax.axvline(x=10, color="red", linestyle="--", alpha=0.5)
    ax.set_title(name, fontsize=11)
    ax.set_ylim(0.5, 6)
    
    # Check if it's a responder: peak after stimulus > 2x baseline std
    baseline_mean = trace[:10].mean()
    baseline_std = trace[:10].std()
    peak = trace[10:].max()
    
    if peak > baseline_mean + 3 * baseline_std:
        ax.text(0.95, 0.95, "Responder", transform=ax.transAxes,
                ha="right", va="top", fontsize=9, color="green", fontweight="bold")
    else:
        ax.text(0.95, 0.95, "Non-responder", transform=ax.transAxes,
                ha="right", va="top", fontsize=9, color="gray")

fig.supxlabel("Time (s)")
fig.supylabel("F / F₀")
fig.suptitle("Calcium Imaging — DRG Neurons + 100 nM Capsaicin (red dashed = stimulus)", 
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

# Quantify: what fraction responded?
n_responders = 0
n_total = len(traces)

print("Neuron-by-neuron analysis:")
print("-" * 50)

for name, trace in traces.items():
    baseline_mean = trace[:10].mean()
    baseline_std = trace[:10].std()
    peak = trace[10:].max()
    peak_time = trace[10:].argmax() + 10  # offset by 10 since we sliced
    delta_f = (peak - baseline_mean) / baseline_mean
    
    is_resp = peak > baseline_mean + 3 * baseline_std
    if is_resp:
        n_responders += 1
    
    status = "RESPONDER" if is_resp else "---"
    print(f"  {name}:  peak F/F0 = {peak:.2f}  ΔF/F0 = {delta_f:.2f}  peak at {peak_time}s  [{status}]")

print(f"\nResponders: {n_responders}/{n_total} ({100*n_responders/n_total:.0f}%)")

This is a realistic analysis workflow: generate or load data (numpy/pandas), visualize it (matplotlib), and quantify results. You just did calcium imaging analysis in Python.

---

## Exercise: Create a bar plot of binder affinities

Using the binder screening CSV from the previous notebook:

1. Read `sample_data/binder_screening.csv` with pandas
2. Create a horizontal bar chart showing each candidate's predicted Kd
3. Color the bars based on whether Kd < 20 nM (good) or >= 20 nM (needs work)
4. Add a vertical dashed line at 20 nM as a threshold marker

**Hints:**
- `plt.barh()` makes horizontal bars
- You can pass a list of colors, one per bar
- `plt.axvline(x=20, ...)` draws a vertical line

In [ ]:
# YOUR CODE HERE
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path



---

### Solution

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

csv_file = Path.cwd() / "sample_data" / "binder_screening.csv"
df = pd.read_csv(csv_file)

# Sort by Kd for a cleaner plot
df = df.sort_values("predicted_kd_nm", ascending=True)

# Color each bar: green if Kd < 20, orange otherwise
colors = ["#22c55e" if kd < 20 else "#f97316" for kd in df["predicted_kd_nm"]]

# Build labels: candidate ID + target
labels = [f"{row['candidate_id']} ({row['target']})" for _, row in df.iterrows()]

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(labels, df["predicted_kd_nm"], color=colors, edgecolor="#374151", alpha=0.85)
ax.axvline(x=20, color="#dc2626", linestyle="--", linewidth=1.5, label="20 nM threshold")
ax.set_xlabel("Predicted Kd (nM)")
ax.set_title("De Novo Binder Screening — Predicted Binding Affinity")
ax.legend()
plt.tight_layout()
plt.show()

---

## Exercise: Multi-neuron calcium trace heatmap

Seaborn can make heatmaps. Create one showing calcium traces for 10 simulated neurons over 30 seconds.

1. Use numpy to generate mock traces (similar to the example above)
2. Put them into a pandas DataFrame (rows = neurons, columns = time points)
3. Use `sns.heatmap()` to visualize

**Hints:**
- `sns.heatmap(df, cmap="YlOrRd")` — the `cmap` argument sets the color scheme
- `YlOrRd` (yellow-orange-red) is good for fluorescence intensity

In [ ]:
# YOUR CODE HERE
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt



---

### Solution

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

np.random.seed(99)
n_neurons = 10
n_timepoints = 30

# Generate traces
all_traces = np.ones((n_neurons, n_timepoints))  # start at baseline = 1.0

for i in range(n_neurons):
    # Add baseline noise
    all_traces[i] += np.random.normal(0, 0.02, n_timepoints)
    
    # ~60% of neurons respond (like a capsaicin challenge on DRG culture)
    if np.random.random() < 0.6:
        peak = np.random.uniform(1.5, 4.5)
        onset = np.random.randint(10, 13)  # slight variation in response onset
        for t in range(onset, n_timepoints):
            all_traces[i, t] += peak * np.exp(-(t - onset) / np.random.uniform(3, 7))

# Build DataFrame
neuron_labels = [f"Neuron {i+1}" for i in range(n_neurons)]
time_labels = [f"{t}s" for t in range(n_timepoints)]
df_traces = pd.DataFrame(all_traces, index=neuron_labels, columns=time_labels)

# Plot heatmap
fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(df_traces, cmap="YlOrRd", ax=ax, 
            xticklabels=5,  # show every 5th time label
            cbar_kws={"label": "F / F₀"})
ax.axvline(x=10, color="white", linewidth=2, linestyle="--")
ax.set_xlabel("Time")
ax.set_ylabel("")
ax.set_title("Calcium Imaging Heatmap — DRG Culture + Capsaicin (white dashed = stimulus)")
plt.tight_layout()
plt.show()

---

## What you just learned

- **Importing packages** — `import`, `from...import`, aliases like `pd`, `np`, `plt`
- **NumPy** — arrays, array math, basic statistics (`.mean()`, `.std()`, `.max()`)
- **Matplotlib** — line plots, bar plots, labeling, formatting
- **Seaborn** — statistical plots, heatmaps, works with DataFrames
- **pandas** — reading CSVs, filtering, grouping, summary stats
- **pip** — installing packages, virtual environments

You now have the foundations to:
1. Read data from files
2. Manipulate it with Python
3. Visualize it with plots

These three skills are the backbone of everything else in this tutorial. Next module, we'll look at how LLMs actually work — which will make you much better at using them.

> **Navigation:** [← Previous: Working with Files](../01-python-foundations/02-working-with-files.ipynb) | [Module Index](../README.md) | [Next: Data Structures in Practice →](../01-python-foundations/04-data-structures-in-practice.ipynb)

---

## Edit Log

- 2026-03-25: Created notebook with initial content
- 2026-03-25: Added "Why this matters" rationale section
- 2026-03-25: Added external references and Further Reading section
- 2026-03-25: Added cross-module navigation links

---

## Edit Log

- 2026-03-25: Created notebook with initial content
- 2026-03-25: Added "Why this matters" rationale section
- 2026-03-25: Added cross-module navigation links